# Grand Mobile · эксперименты классификатора категорий (v2)

Отбор — **5-fold CV на train+dev** (668 строк; синтетика только в обучении, метрика — по реальным тикетам).
Сетка: TF-IDF (char / char+word) × энкодеры (E5 small/base/large, BGE-M3, замороженные) × признаки регулярок репо × C.
Дообучение E5-base на 2×T4 — отдельный эксперимент на dev. **Test (люди) — один раз в конце** для победителя и baseline.
Kaggle: Private dataset `gm_nlp_data.jsonl`, GPU T4 x2, Internet On, Run All.

In [ ]:
import os, glob, json, hashlib, time, random, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
from joblib import Parallel, delayed
from scipy.sparse import hstack, csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

SMOKE = os.environ.get("GM_SMOKE") == "1"
SEED = 20261031; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"; N_GPU = torch.cuda.device_count()
print(DEVICE, N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
assert SMOKE or N_GPU > 0, "GPU is off: Settings -> Accelerator -> GPU T4 x2, then restart the session"
# name -> (pooling, prefix)
ENCODERS = {"intfloat/multilingual-e5-small": ("mean", "query: "), "intfloat/multilingual-e5-base": ("mean", "query: "),
            "intfloat/multilingual-e5-large": ("mean", "query: "), "BAAI/bge-m3": ("cls", "")}
if SMOKE: ENCODERS = {os.environ["GM_LOCAL_E5"]: ("mean", "query: ")}
C_GRID = [4, 16, 64] if not SMOKE else [16]
RUN_FINETUNE = not SMOKE

In [ ]:
path = os.environ.get("GM_DATA") or glob.glob("/kaggle/input/**/gm_nlp_data.jsonl", recursive=True)[0]
df = pd.read_json(path, lines=True); DATA_SHA = hashlib.sha256(open(path, "rb").read()).hexdigest()[:16]
CATS = {"complaint": ["complaint.insult", "complaint.cheat", "complaint.dm", "complaint.scam", "complaint.rmt_ads",
                      "complaint.admin", "complaint.other", "complaint.invalid"],
        "report": ["report.question", "report.help", "report.complaint", "report.account", "report.payment", "report.offtopic"]}
LABELS = CATS["complaint"] + CATS["report"]; PRIOS = ["p0", "p1", "p2", "p3"]
DEFAULT_PRIO = {"complaint.insult": "p2", "complaint.cheat": "p1", "complaint.dm": "p2", "complaint.scam": "p1",
                "complaint.rmt_ads": "p1", "complaint.admin": "p2", "complaint.other": "p2", "complaint.invalid": "p3",
                "report.question": "p3", "report.help": "p2", "report.complaint": "p2", "report.account": "p2",
                "report.payment": "p0", "report.offtopic": "p3"}
def model_input(r):
    return " | ".join([f"канал: {r.kind}"] + ([f"пункт правил: {r.field}"] if r.field else []) + [r.text[:2000]])
df["input"] = [model_input(r) for r in df.itertuples()]
cv_df = df[df.split.isin(["train", "dev"])].reset_index(drop=True)      # model selection pool
te = df[df.split == "test"].reset_index(drop=True)                       # touched once, at the end
REAL = ~cv_df.id.str.startswith("syn").values                            # score real tickets only
print(DATA_SHA, len(cv_df), "(real", REAL.sum(), ")", len(te))

In [ ]:
class Pooled(torch.nn.Module):
    def __init__(self, name, pooling):
        super().__init__(); self.enc = AutoModel.from_pretrained(name); self.pooling = pooling
    def forward(self, input_ids, attention_mask):
        h = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        if self.pooling == "cls": v = h[:, 0]
        else:
            m = attention_mask.unsqueeze(-1).to(h.dtype); v = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
        return torch.nn.functional.normalize(v, dim=-1)

@torch.inference_mode()
def embed(name, texts, batch=64):
    pooling, prefix = ENCODERS[name]; tok = AutoTokenizer.from_pretrained(name)
    net = Pooled(name, pooling).to(DEVICE).eval()
    if DEVICE == "cuda": net = net.half()
    if N_GPU > 1: net = torch.nn.DataParallel(net)                       # both T4
    out = []
    for i in range(0, len(texts), batch):
        b = tok([prefix + t for t in texts[i:i + batch]], max_length=512, truncation=True, padding=True, return_tensors="pt").to(DEVICE)
        out.append(net(b["input_ids"], b["attention_mask"]).float().cpu())
    del net; torch.cuda.empty_cache(); return torch.cat(out).numpy()

EMB = {}
for name in ENCODERS:
    t0 = time.time(); EMB[name] = {"cv": embed(name, cv_df.input.tolist()), "test": embed(name, te.input.tolist())}
    print(f"{name}: {EMB[name]['cv'].shape} {time.time() - t0:.0f}s")

In [ ]:
REG = sorted(set(df.baseline_category))
def reg_onehot(d): return csr_matrix(np.array([[c == x for x in REG] for c in d.baseline_category], dtype=np.float32))
def masked(proba, classes, kinds):
    classes = list(classes); preds, conf = [], []
    for p, k in zip(proba, kinds):
        al = [i for i, c in enumerate(classes) if c.startswith(k + ".")]
        if not al: preds.append(CATS[k][0]); conf.append(0.0); continue
        i = max(al, key=lambda j: p[j]); s = p[al].sum(); preds.append(classes[i]); conf.append(float(p[i] / s) if s else 0.0)
    return np.array(preds), np.array(conf)

TFIDF = {"charwb25": [("char_wb", (2, 5))], "char+word": [("char_wb", (1, 5)), ("word", (1, 2))]}
def build(cfg, fit, app, fit_idx, app_idx, split_app="cv"):
    """cfg = (tfidf_key|None, encoder|None, use_regex). Returns fitted feature matrices and the vectorizers."""
    tf_key, enc, use_reg = cfg; F, A, vecs = [], [], []
    if tf_key:
        for a, n in TFIDF[tf_key]:
            v = TfidfVectorizer(analyzer=a, ngram_range=n, sublinear_tf=True, max_features=150_000)
            F.append(v.fit_transform(fit.input)); A.append(v.transform(app.input)); vecs.append(v)
    if enc:
        F.append(csr_matrix(EMB[enc]["cv"][fit_idx] * 2.0))
        A.append(csr_matrix((EMB[enc]["cv"][app_idx] if split_app == "cv" else EMB[enc]["test"]) * 2.0))
    if use_reg: F.append(reg_onehot(fit)); A.append(reg_onehot(app))
    return hstack(F).tocsr(), hstack(A).tocsr(), vecs

def to_dev(m, device):
    # scipy sparse -> dense float32 tensor built on the target device (no dense copy on the CPU).
    m = m.tocsr().astype(np.float32)
    t = torch.sparse_csr_tensor(torch.from_numpy(m.indptr).long(), torch.from_numpy(m.indices).long(),
                                torch.from_numpy(m.data), size=m.shape)
    return t.to(device).to_dense()

class TorchLR:
    """sklearn-equivalent L2 multinomial logistic regression with balanced class weights (full-batch L-BFGS):
    minimise sum_i w_i * CE_i + ||W||^2 / (2C). Same predictions as sklearn on this data (checked);
    the final model is refit with sklearn so the artifact runs on CPU without torch."""
    def __init__(self, C, device): self.C, self.device = C, device
    def fit_tensor(self, Xt, y):
        y = np.asarray(y); self.classes_ = np.array(sorted(set(y))); k = len(self.classes_)
        yi = torch.tensor(np.searchsorted(self.classes_, y), device=self.device)
        cnt = np.bincount(yi.cpu().numpy(), minlength=k)
        sw = torch.tensor(len(y) / (k * cnt), device=self.device, dtype=torch.float32)[yi]
        self.W = torch.zeros(Xt.shape[1], k, device=self.device, requires_grad=True)
        self.b = torch.zeros(k, device=self.device, requires_grad=True)
        opt = torch.optim.LBFGS([self.W, self.b], lr=1, max_iter=300, history_size=20, tolerance_grad=1e-6, line_search_fn="strong_wolfe")
        def closure():
            opt.zero_grad()
            loss = (torch.nn.functional.cross_entropy(Xt @ self.W + self.b, yi, reduction="none") * sw).sum() + (self.W ** 2).sum() / (2 * self.C)
            loss.backward(); return loss
        opt.step(closure); return self
    @torch.inference_mode()
    def proba_tensor(self, Xt): return torch.softmax(Xt @ self.W + self.b, -1).cpu().numpy()
    def fit(self, X, y): return self.fit_tensor(to_dev(X, self.device), y)
    def predict_proba(self, X): return self.proba_tensor(to_dev(X, self.device))
    def predict(self, X): return self.classes_[self.predict_proba(X).argmax(1)]

def make_lr(C, device):
    return TorchLR(C, device) if device.startswith("cuda") else LogisticRegression(C=C, max_iter=5000, class_weight="balanced")

def cv_score(cfg, C, target="category", device=DEVICE):
    # Single-config CV (used for the priority head).
    pred = np.empty(len(cv_df), dtype=object)
    for a, b in FOLDS:
        Xf, Xa, _ = build(cfg, cv_df.iloc[a], cv_df.iloc[b], a, b)
        clf = make_lr(C, device).fit(Xf, cv_df[target].iloc[a])
        pred[b] = (masked(clf.predict_proba(Xa), clf.classes_, cv_df.kind.iloc[b])[0] if target == "category" else clf.predict(Xa))
    y = cv_df[target].values[REAL]; p = pred[REAL]
    return {"cfg": cfg, "C": C, "target": target, "macro_f1": round(f1_score(y, p, average="macro", zero_division=0), 4),
            "acc": round(accuracy_score(y, p), 4)}

FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=0).split(cv_df, cv_df.category))
grid = [(t, e, r) for t in [None, *TFIDF] for e in [None, *ENCODERS] for r in (False, True) if t or e]
devices = [f"cuda:{i}" for i in range(N_GPU)] or ["cpu"]
print(len(grid) * len(C_GRID), "configs x 5 folds on", devices)

def run_fold(fi, device):
    # Features of one fold are built once and kept on the GPU; every config and C reuses them.
    a, b = FOLDS[fi]; fit, app = cv_df.iloc[a], cv_df.iloc[b]; blocks = {}
    for key, specs in TFIDF.items():
        F, A = [], []
        for an, n in specs:
            v = TfidfVectorizer(analyzer=an, ngram_range=n, sublinear_tf=True, max_features=150_000)
            F.append(to_dev(v.fit_transform(fit.input), device)); A.append(to_dev(v.transform(app.input), device))
        blocks[key] = (torch.cat(F, 1), torch.cat(A, 1))
    for enc in ENCODERS:
        blocks[enc] = (torch.tensor(EMB[enc]["cv"][a] * 2.0, device=device, dtype=torch.float32),
                       torch.tensor(EMB[enc]["cv"][b] * 2.0, device=device, dtype=torch.float32))
    blocks["reg"] = (to_dev(reg_onehot(fit), device), to_dev(reg_onehot(app), device))
    out = {}
    for cfg in grid:
        keys = [k for k in (cfg[0], cfg[1], "reg" if cfg[2] else None) if k]
        Xf = torch.cat([blocks[k][0] for k in keys], 1); Xa = torch.cat([blocks[k][1] for k in keys], 1)
        for C in C_GRID:
            clf = TorchLR(C, device).fit_tensor(Xf, fit.category.values)
            out[(cfg, C)] = masked(clf.proba_tensor(Xa), clf.classes_, app.kind)[0]
        del Xf, Xa
    del blocks; torch.cuda.empty_cache() if device.startswith("cuda") else None
    print(f"fold {fi} done on {device}", flush=True)
    return fi, out

t0 = time.time()
fold_out = Parallel(n_jobs=len(devices), backend="threading")(delayed(run_fold)(fi, devices[fi % len(devices)]) for fi in range(len(FOLDS)))
oof = {key: np.empty(len(cv_df), dtype=object) for key in fold_out[0][1]}
for fi, out in fold_out:
    for key, pred in out.items(): oof[key][FOLDS[fi][1]] = pred
y = cv_df.category.values[REAL]
rows = [{"cfg": cfg, "C": C, "target": "category", "macro_f1": round(f1_score(y, p[REAL], average="macro", zero_division=0), 4),
         "acc": round(accuracy_score(y, p[REAL]), 4)} for (cfg, C), p in oof.items()]
print(f"{time.time() - t0:.0f}s")
y = cv_df.category.values[REAL]
rows.append({"cfg": ("baseline_regex",), "C": None, "target": "category",
             "macro_f1": round(f1_score(y, cv_df.baseline_category.values[REAL], average="macro", zero_division=0), 4),
             "acc": round(accuracy_score(y, cv_df.baseline_category.values[REAL]), 4)})
board = pd.DataFrame(rows).sort_values(["macro_f1", "acc"], ascending=False).reset_index(drop=True)
board["cfg"] = board.cfg.astype(str); board.head(20)

In [ ]:
# Fine-tune experiment (dev holdout only: CV would be 5x the GPU time). Separate LRs for encoder/head, warmup, more steps.
FT_DEV = None
if RUN_FINETUNE:
    trn, dv = cv_df[cv_df.split == "train"].reset_index(drop=True), cv_df[cv_df.split == "dev"].reset_index(drop=True)
    name = "intfloat/multilingual-e5-base"; tok = AutoTokenizer.from_pretrained(name); idx = {c: i for i, c in enumerate(LABELS)}
    class Clf(torch.nn.Module):
        def __init__(self):
            super().__init__(); self.pool = Pooled(name, "mean"); self.drop = torch.nn.Dropout(0.1)
            self.head = torch.nn.Linear(self.pool.enc.config.hidden_size, len(LABELS))
        def forward(self, ids, mask): return self.head(self.drop(self.pool(ids, mask)))
    net = Clf().to(DEVICE); para = torch.nn.DataParallel(net) if N_GPU > 1 else net
    EPOCHS, BS = 15, 16
    opt = torch.optim.AdamW([{"params": net.pool.parameters(), "lr": 2e-5}, {"params": net.head.parameters(), "lr": 1e-3}], weight_decay=0.01)
    steps = EPOCHS * ((len(trn) + BS - 1) // BS); sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler("cuda"); cnt = trn.category.value_counts()
    lossf = torch.nn.CrossEntropyLoss(weight=torch.tensor([len(trn) / (len(cnt) * cnt.get(c, len(trn))) for c in LABELS], device=DEVICE).float(), label_smoothing=0.05)
    def batches(d, bs, shuffle):
        order = np.random.permutation(len(d)) if shuffle else np.arange(len(d))
        for i in range(0, len(d), bs):
            part = d.iloc[order[i:i + bs]]
            b = tok(["query: " + t for t in part.input], max_length=256, truncation=True, padding=True, return_tensors="pt")
            yield b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE), part
    @torch.inference_mode()
    def ft_predict(d):
        para.eval(); P = []
        for ids, mask, _ in batches(d, 64, False):
            with torch.autocast("cuda", dtype=torch.float16): P.append(torch.softmax(para(ids, mask).float(), -1).cpu())
        return masked(torch.cat(P).numpy(), LABELS, d.kind)
    best, state, bad = -1, None, 0
    for ep in range(EPOCHS):
        para.train()
        for ids, mask, part in batches(trn, BS, True):
            yt = torch.tensor([idx[c] for c in part.category], device=DEVICE)
            with torch.autocast("cuda", dtype=torch.float16): loss = lossf(para(ids, mask).float(), yt)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
        f1 = f1_score(dv.category, ft_predict(dv)[0], average="macro", zero_division=0); print(ep, round(f1, 4))
        if f1 > best: best, bad, state = f1, 0, {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= 4: break
    net.load_state_dict(state); FT_DEV = round(best, 4)
    # Same dev holdout for the best CV config, so the two are comparable.
    top = board[board.C.notna()].iloc[0]; cfg0, C0 = eval(top.cfg), top.C
    ti = np.where(cv_df.split == "train")[0]; di = np.where(cv_df.split == "dev")[0]
    Xf, Xa, _ = build(cfg0, trn, dv, ti, di)
    lr0 = make_lr(C0, DEVICE if N_GPU == 0 else "cuda:0").fit(Xf, trn.category)
    p0 = masked(lr0.predict_proba(Xa), lr0.classes_, dv.kind)[0]
    print({"finetuned_e5_base_dev_f1": FT_DEV, "best_cv_config_dev_f1": round(f1_score(dv.category, p0, average="macro", zero_division=0), 4)})

In [ ]:
# Priority: default-by-category (from CV predictions of the best config) vs learned head, by CV.
top = board[board.C.notna()].iloc[0]; cfg_best, C_best = eval(top.cfg), top.C
skf = StratifiedKFold(5, shuffle=True, random_state=0); cat_oof = np.empty(len(cv_df), dtype=object)
for a, b in skf.split(cv_df, cv_df.category):
    Xf, Xa, _ = build(cfg_best, cv_df.iloc[a], cv_df.iloc[b], a, b)
    clf = make_lr(C_best, DEVICE if N_GPU == 0 else "cuda:0").fit(Xf, cv_df.category.iloc[a])
    cat_oof[b] = masked(clf.predict_proba(Xa), clf.classes_, cv_df.kind.iloc[b])[0]
yp = cv_df.priority.values[REAL]
prio = {"default_by_category": round(f1_score(yp, np.array([DEFAULT_PRIO[c] for c in cat_oof])[REAL], average="macro", zero_division=0), 4),
        "learned_head": cv_score(cfg_best, C_best, "priority")["macro_f1"],
        "baseline_regex": round(f1_score(yp, cv_df.baseline_priority.values[REAL], average="macro", zero_division=0), 4)}
PRIO_METHOD = max(prio, key=prio.get); print(prio, "->", PRIO_METHOD)

In [ ]:
# ===== FINAL: winner refit on all train+dev, test scored once =====
winner_cfg, winner_C = cfg_best, C_best
all_idx = np.arange(len(cv_df))
Xf, Xt, vecs = build(winner_cfg, cv_df, te, all_idx, None, split_app="test")
clf = LogisticRegression(C=winner_C, max_iter=5000, class_weight="balanced").fit(Xf, cv_df.category)
p_te, c_te = masked(clf.predict_proba(Xt), clf.classes_, te.kind)
prio_clf = None
if PRIO_METHOD == "learned_head":
    prio_clf = LogisticRegression(C=winner_C, max_iter=5000, class_weight="balanced").fit(Xf, cv_df.priority); pr_te = prio_clf.predict(Xt)
elif PRIO_METHOD == "default_by_category": pr_te = np.array([DEFAULT_PRIO[c] for c in p_te])
else: pr_te = te.baseline_priority.values
L = sorted(set(te.category))
final = pd.DataFrame([
    {"model": "baseline_regex", "acc": accuracy_score(te.category, te.baseline_category),
     "macro_f1": f1_score(te.category, te.baseline_category, labels=L, average="macro", zero_division=0),
     "priority_macro_f1": f1_score(te.priority, te.baseline_priority, labels=PRIOS, average="macro", zero_division=0)},
    {"model": f"{winner_cfg} C={winner_C}", "acc": accuracy_score(te.category, p_te),
     "macro_f1": f1_score(te.category, p_te, labels=L, average="macro", zero_division=0),
     "priority_macro_f1": f1_score(te.priority, pr_te, labels=PRIOS, average="macro", zero_division=0)}]).round(3)
print(f"TEST n={len(te)} (human labels)"); display(final)
print(classification_report(te.category, p_te, labels=L, zero_division=0))
te["pred"] = p_te; te["lang_bucket"] = te.lang_hint.where(te.lang_hint.isin(["ru", "uz", "kk"]), "other")
display(te.groupby("lang_bucket").apply(lambda g: pd.Series({"n": len(g), "acc": round((g.pred == g.category).mean(), 3)})))

In [ ]:
import joblib
out_dir = "/kaggle/working" if not SMOKE else "smoke_out"; os.makedirs(out_dir, exist_ok=True)
tf_key, enc, use_reg = winner_cfg
art = {"version": f"gm-cat-{DATA_SHA}", "config": {"tfidf": tf_key, "encoder": enc, "regex_onehot": use_reg, "C": winner_C,
       "encoder_weight": 2.0, "input_format": "канал: <kind> | пункт правил: <field> | <text[:2000]>"},
       "vectorizers": vecs, "regex_categories": REG, "head": clf, "labels": LABELS, "channel_labels": CATS,
       "priority_method": PRIO_METHOD, "priority_head": prio_clf, "default_priority": DEFAULT_PRIO,
       "cv_board_top10": board.head(10).to_dict("records"), "finetune_dev_f1": FT_DEV, "metrics_test": final.to_dict("records"),
       "confidence_kind": "uncalibrated", "data_sha256_16": DATA_SHA}
joblib.dump(art, f"{out_dir}/gm_category_model_v2.joblib")
board.to_csv(f"{out_dir}/cv_board.csv", index=False)
json.dump({k: v for k, v in art.items() if k not in ("vectorizers", "head", "priority_head")}, open(f"{out_dir}/gm_model_card_v2.json", "w"),
          ensure_ascii=False, indent=1, default=str)
print("saved", out_dir)